# Data completeness analysis

Cross-cutting layer over the three profile projects. Completeness in this data is not
one score; it decomposes into five signals: core presence, the SS gate, series
integrity (holes, alignment, continuity), cross-source consistency flags, and
analysis readiness. The first three of the original signals (measured on the 565 snapshots):

1. **Core sources** (CRC, personal data, e-fatura, IRS, debt certificate, BCB accounts):
   95-100% coverage. A miss is an extraction defect: flag and refetch.
2. **The Social Security block** (salary, SS payments, calculated incomes, job info)
   opens as one gate: co-occurrence 94-98% given salary. Measured against the other
   vectors, the gate is a **data-access artifact, not a personal trait**: both sides
   show the same self-employment rates, incomes and significant-default rates.
3. **Situational sources** (activity, invoices, renter contracts, pension): **at most
   weakly descriptive**. Renter presence correlates with not holding a mortgage; the
   rest showed no measurable person-level meaning, and missingness barely predicts
   default (AUC 0.51-0.53). Evidence in the spec's 'Measured meaning of absence'.

Module: `completeness_features.py`. Output: `completeness_layers_output.csv`.

In [ ]:
import json
import sys

import pandas as pd

sys.path.insert(0, ".")
from completeness_features import (CORE_SOURCES, SS_BLOCK, completeness_features,
                                   render_profile_text)

pd.set_option("display.width", 200)
raw = pd.read_csv("../data/data_full.csv")
snap = raw.drop_duplicates("uuid")[["uuid", "value"]].reset_index(drop=True)
feats = pd.DataFrame([
    {"uuid": u, **completeness_features(json.loads(v))}
    for u, v in zip(snap["uuid"], snap["value"])
])
print(f"{feats.shape[0]} proponents x {feats.shape[1] - 1} fields, "
      f"version {feats['features_version'].iloc[0]}")

## 1. The three signals across the population

In [ ]:
print("core coverage:")
print((feats[[f"has_{s}" for s in CORE_SOURCES]].mean() * 100).round(1).to_string())
print(f"\ncore_complete: {feats['core_complete'].sum()} of {len(feats)} "
      f"({feats['core_complete'].mean()*100:.1f}%)")
print("missing_core breakdown:")
print(feats.loc[feats["missing_core"].notna(), "missing_core"].value_counts().to_string())

print("\nSS block co-occurrence (given has_net_salary):")
sal = feats[feats["has_net_salary"] == 1]
for s in SS_BLOCK:
    print(f"  has_{s}: {sal[f'has_{s}'].mean()*100:.0f}%")

print("\nincome_verification regimes:")
print(feats["income_verification"].value_counts().to_string())

print("\nsituational presence:")
for s in ("activity_registration", "invoices", "renter_contracts", "pension"):
    print(f"  has_{s}: {int(feats[f'has_{s}'].sum())}")

## 2b. Series integrity and consistency flags (v1.1.0)

Depth beyond presence: holes inside monthly series, salary/SS alignment, income-vs-life
continuity, and cross-source contradictions.

In [ ]:
print("series holes (proponents with months missing inside the span):")
for c in ("salary_gap_months", "ss_gap_months", "receipt_gap_months"):
    s = feats.loc[feats[c].fillna(0) > 0, c]
    print(f"  {c}: {len(s)} proponents, median {s.median() if len(s) else 0:.0f}, "
          f"max {s.max() if len(s) else 0:.0f}")

print("\nconsistency flag prevalence:")
from collections import Counter
counter = Counter()
for flags in feats["consistency_flags"].dropna():
    for fl in flags.split(";"):
        counter[fl] += 1
for fl, n in counter.most_common():
    print(f"  {fl}: {n}")
print(f"\nproponents with 0 flags: {(feats['n_consistency_flags'] == 0).sum()}, "
      f"1 flag: {(feats['n_consistency_flags'] == 1).sum()}, "
      f"2+: {(feats['n_consistency_flags'] >= 2).sum()}")

print("\nanalysis readiness (n_analyses_possible of 5):")
print(feats["n_analyses_possible"].value_counts().sort_index().to_string())

## 2c. Extraction telemetry (v1.2.0)

ServicesResults resolves WHY data is absent: 484 = no data exists (validated exactly
against the rent and terminations sources); other never-recovered codes = technical
failure; services can also simply not be invoked (the SS gate's actual cause).

In [ ]:
print("ss_gate_cause:")
print(feats["ss_gate_cause"].value_counts().to_string())
print("\nper-source fetch status:")
for c in ("rent_fetch", "invoices_fetch", "terminations_fetch", "activity_fetch"):
    print(f"  {c}: {feats[c].value_counts().to_dict()}")
print("\nrefetch recommended:", int(feats["refetch_recommended"].fillna(0).sum()))
from collections import Counter
fails = Counter()
for s in feats["failed_services"].dropna():
    for name in s.split(";"):
        fails[name] += 1
print("failing services:", dict(fails.most_common(8)))
print("\nlandlords (rental income exists):", int((feats["is_landlord"] == 1).sum()))

## 2. Freshness

Staleness is measured against `data_anchor_month` (the newest month observed anywhere
in that snapshot), so it is extraction-relative, not wall-clock.

In [ ]:
stale_cols = ["efatura_staleness_months", "salary_staleness_months",
              "ss_staleness_months", "crc_staleness_months"]
print(feats[stale_cols].describe(percentiles=[.5, .9]).round(1).T.to_string())
print("\nsalary more than 3 months staler than the anchor:",
      int((feats["salary_staleness_months"] > 3).sum()))

## 3. Layer 2: the data-quality strip

One example per regime, plus an incomplete-core case.

In [ ]:
fset = feats.set_index("uuid")
vmap = dict(zip(snap["uuid"], snap["value"]))
examples = [
    fset[(fset["income_verification"] == "ss_verified") & (fset["core_complete"] == 1)].index[0],
    fset[(fset["income_verification"] == "irs_only") & (fset["core_complete"] == 1)].index[0],
    fset[fset["income_verification"] == "none"].index[0],
    fset[fset["core_complete"] == 0].index[0],
]
for u in examples:
    print(f"=== {u[:8]} ===")
    print(render_profile_text(completeness_features(json.loads(vmap[u]))))
    print()

## 4. Validation: module vs independent recomputation. Must print 0.

In [ ]:
def dig(d, *path):
    for k in path:
        d = d.get(k) if isinstance(d, dict) else None
    return d

mism = checked = 0
for uuid, v in zip(snap["uuid"], snap["value"]):
    p = json.loads(v)
    vec = completeness_features(p)
    irs_c = dig(p, "IRSTaxDeclarationDetails", "Total", "Income")
    irs_p = dig(p, "IRSTaxDeclarationDetailsPreviousYear", "Total", "Income")
    has_irs = (irs_c is not None and irs_c >= 1) or (irs_p is not None and irs_p >= 1)
    expect = {
        "has_crc": bool(dig(p, "CRCData", "Liabilities")),
        "has_efatura": bool(p.get("ExpensesList")),
        "has_irs": has_irs,
        "has_ss_block": bool(p.get("NetSalary")) or bool(dig(p, "SSPayments", "Payments")),
        "core_complete": all([
            bool(dig(p, "CRCData", "Liabilities")),
            dig(p, "PersonalTaxData", "BirthDate") is not None,
            bool(p.get("ExpensesList")), has_irs,
            dig(p, "IRSNonDebtCertificateData", "HasDebts") is not None,
            bool(dig(p, "BCBData", "Accounts")),
        ]),
    }
    for k, e in expect.items():
        checked += 1
        if vec[k] != int(e):
            mism += 1
            print("MISMATCH", uuid[:8], k, vec[k], int(e))
print(f"{checked} comparisons, mismatches: {mism}")